# Inferencia con Groq

**Los mismos modelos abiertos, ahora servidos en la nube a alta velocidad.** En la lección 1
corrimos un LLM open source en tu máquina con Ollama; aquí consumimos modelos **open-weights**
(los `gpt-oss` de OpenAI) servidos por **[Groq](https://groq.com)**, cuyo hardware
especializado (LPU) alcanza cientos de tokens por segundo.

Pasos:

1. Crea una cuenta gratuita y una API key en https://console.groq.com.
2. Guárdala como `GROQ_API_KEY` — en Colab en los secretos (`userdata`), en local en `.env`
   (ver README).
3. *(Opcional)* define `LANGSMITH_API_KEY` si quieres tracing en LangSmith; si no existe,
   el notebook no lo activa.


In [ ]:
# En Colab: instala las dependencias fijadas. Local: usa el entorno uv del README.
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "langchain-groq==1.1.3", "langchain-core==1.4.9",
        "langsmith==0.9.5", "python-dotenv==1.2.2",
    ])
    print("Dependencias instaladas (Colab)")
else:
    print("Entorno local: usando las dependencias del entorno uv (ver README)")


In [ ]:
import os
from pathlib import Path


def load_keys() -> None:
    """Carga GROQ_API_KEY (requerida) y LANGSMITH_API_KEY (opcional) desde Colab o .env."""
    if IN_COLAB:
        from google.colab import userdata

        try:
            os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
        except Exception as exc:
            raise ValueError(
                "Define GROQ_API_KEY en los secretos de Colab (ícono de llave, "
                "con acceso al notebook activado)."
            ) from exc
        try:
            langsmith_key = userdata.get("LANGSMITH_API_KEY")
            if langsmith_key:
                os.environ["LANGSMITH_API_KEY"] = langsmith_key
        except Exception:
            pass  # la llave de LangSmith es opcional
        print("Llaves cargadas desde Colab userdata")
    else:
        from dotenv import load_dotenv

        load_dotenv(Path(".env"))
        print("Variables cargadas desde .env (si existe)")

    if not os.environ.get("GROQ_API_KEY"):
        raise ValueError("Falta GROQ_API_KEY — créala gratis en https://console.groq.com")

    # Tracing en LangSmith SOLO si hay llave; si no, queda desactivado (sin warnings).
    if os.environ.get("LANGSMITH_API_KEY"):
        os.environ.setdefault("LANGSMITH_TRACING", "true")
        os.environ.setdefault("LANGSMITH_PROJECT", "clase-5-1-groq")
        print(f"Tracing LangSmith activo (proyecto: {os.environ['LANGSMITH_PROJECT']})")
    else:
        print("Sin LANGSMITH_API_KEY — tracing desactivado (es opcional)")


load_keys()


## Los modelos `gpt-oss`

Los [`gpt-oss`](https://console.groq.com/docs/models) son los modelos **open-weights de
OpenAI** (20B y 120B parámetros) servidos por Groq. Usamos la misma traducción ES→EN de la
lección 1 y medimos la **velocidad real** de cada modelo con los metadatos que devuelve la API
(`token_usage.completion_tokens / completion_time`).

> Groq rota su catálogo: si un modelo deja de estar disponible, elige otro en
> https://console.groq.com/docs/models (p. ej. `llama-3.3-70b-versatile`).


In [ ]:
from langchain_groq import ChatGroq

messages = [
    ("system", "Eres un traductor. Traduce la frase del usuario del español al inglés."),
    ("human", "Me encanta programar."),
]


def consultar(model_name, messages):
    """Invoca un modelo servido por Groq y calcula su velocidad real (tokens/segundo)."""
    llm = ChatGroq(model=model_name, temperature=0, max_retries=2)
    respuesta = llm.invoke(messages)
    uso = respuesta.response_metadata["token_usage"]
    velocidad = uso["completion_tokens"] / uso["completion_time"]
    print(f"[{model_name}]")
    print(f"Respuesta: {respuesta.content}")
    print(f"Velocidad: {velocidad:.0f} t/s")
    return velocidad


In [ ]:
velocidad_20b = consultar("openai/gpt-oss-20b", messages)


In [ ]:
velocidad_120b = consultar("openai/gpt-oss-120b", messages)


In [ ]:
print("Modelo               | Velocidad")
print("---------------------|------------")
print(f"openai/gpt-oss-20b   | {velocidad_20b:7.0f} t/s")
print(f"openai/gpt-oss-120b  | {velocidad_120b:7.0f} t/s")
print()
print("Compara con la lección 1: un modelo abierto local corre a ~20–80 t/s en un laptop.")


## Cierre

Un mismo modelo open source puede consumirse donde más convenga: **local** con Ollama
(lección 1 — privacidad y costo cero), **afinado a tus datos** (lección 2 — fine-tuning) o
**servido en la nube especializada** como Groq (esta lección — cientos de t/s, pago por token).
Elegir dónde corre el modelo es una decisión de producto, no solo técnica.
